# 🎵 Emotion-Aware Music Chatbot — Data Cleaning Pipeline v2

**Project:** An NLP-Based Chatbot for Empathetic Emotional Support and Mood-Based Music Playlist Recommendation  
**Target:** Journal MIND — SINTA 3, ITENAS Bandung  

---

## 🎯 Preprocessing Standard

Setiap dataset menghasilkan **satu kolom `final_text`** yang melewati pipeline preprocessing lengkap secara konsisten:

```
Raw Text
  → [1] Lowercase
  → [2] Remove URL, HTML, special characters, extra whitespace
  → [3] Expand contractions
  → [4] Remove punctuation
  → [5] Stopwords Removal (emotion-safe)
  → [6] Lemmatization
  → final_text
```

| # | Dataset | `final_text` Sumber | Output File |
|---|---------|---------------------|-------------|
| 1 | `goemotions.csv` | kolom `text` | `cleaned_goemotions.csv` |
| 2 | `empathetic_dialogues.csv` | kolom `Situation` (sebagai prompt) | `cleaned_empathetic_dialogues.csv` |
| 3 | `spotify_lyrics.csv` | kolom `lyrics` | `cleaned_spotify_lyrics.csv` |

> **Catatan:** Kolom `clean_text` (sebelum stopwords/lemmatization) tetap disimpan untuk kebutuhan RoBERTa yang memiliki tokenizer sendiri.

---
## ⚙️ 0. Install & Import

In [ ]:
!pip install nltk langdetect -q

import pandas as pd
import numpy as np
import re
import os
import json
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from langdetect import detect, LangDetectException

nltk.download('punkt',       quiet=True)
nltk.download('punkt_tab',   quiet=True)
nltk.download('stopwords',   quiet=True)
nltk.download('wordnet',     quiet=True)
nltk.download('omw-1.4',     quiet=True)

os.makedirs('/kaggle/working', exist_ok=True)
print('✅ Dependencies loaded.')

---
## 🔧 1. Shared Preprocessing Pipeline

Fungsi-fungsi ini digunakan **konsisten** di ketiga dataset untuk menghasilkan `final_text`.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────
lemmatizer = WordNetLemmatizer()
BASE_STOPWORDS = set(stopwords.words('english'))

# Kata-kata yang DIPERTAHANKAN meskipun termasuk stopword
# karena membawa sinyal emosi yang penting untuk klasifikasi
EMOTION_PRESERVE = {
    'not', 'no', 'never', 'neither', 'nor',
    'very', 'too', 'so', 'really', 'quite',
    'just', 'only', 'more', 'most',
    'against', 'up', 'down', 'out', 'off',
    'won', 'can', 'did', 'is', 'are', 'was',
    'all', 'both', 'why', 'how'
}
FINAL_STOPWORDS = BASE_STOPWORDS - EMOTION_PRESERVE

print(f'Base stopwords : {len(BASE_STOPWORDS)}')
print(f'After preserve : {len(FINAL_STOPWORDS)} (removed {len(BASE_STOPWORDS)-len(FINAL_STOPWORDS)} emotion words)')

In [ ]:
#Mengubah kata ke bentuk dasar dengan mempertimbangkan POS/ Part Of Speech
def _get_wordnet_pos(word: str) -> str:
    """Return WordNet POS tag; default to noun."""
    # Simple verb list covering common lemmatizer failures
    _verb_set = {
        'was', 'were', 'has', 'have', 'had', 'is', 'are',
        'does', 'did', 'goes', 'went', 'comes', 'came',
        'says', 'said', 'gets', 'got', 'makes', 'made',
        'takes', 'took', 'knows', 'knew', 'thinks', 'thought',
        'feels', 'felt', 'seems', 'seemed', 'looks', 'looked',
        'runs', 'ran', 'tries', 'tried', 'wants', 'wanted',
        'needs', 'needed', 'keeps', 'kept', 'starts', 'started',
    }
    return wordnet.VERB if word in _verb_set else wordnet.NOUN

In [ ]:
# ── Step-by-step functions ─────────────────────────────────────────────

def step1_lowercase(text: str) -> str:
    """Step 1: Lowercase semua karakter."""
    if not isinstance(text, str):
        return ''
    return text.lower().strip()


def step2_remove_noise(text: str) -> str:
    """Step 2: Hapus URL, HTML tags, mention, hashtag, angka, extra whitespace."""
    text = re.sub(r'http\S+|www\.\S+',  '', text)   # URL
    text = re.sub(r'<[^>]+>',            '', text)   # HTML tags
    text = re.sub(r'@\w+',               '', text)   # @mention
    text = re.sub(r'#\w+',               '', text)   # #hashtag
    text = re.sub(r'\d+',                '', text)   # angka
    text = re.sub(r'\s+',               ' ', text)   # whitespace ganda
    return text.strip()


def step3_expand_contractions(text: str) -> str:
    """Step 3: Expand kontraksi Bahasa Inggris."""
    contractions = {
        # Negations — full word forms first
        "won't":    "will not",   "can't":     "cannot",
        "couldn't": "could not",  "wouldn't":  "would not",
        "shouldn't":"should not", "didn't":    "did not",
        "doesn't":  "does not",   "don't":     "do not",
        "isn't":    "is not",     "aren't":    "are not",
        "wasn't":   "was not",    "weren't":   "were not",
        "haven't":  "have not",   "hasn't":    "has not",
        "hadn't":   "had not",    "mustn't":   "must not",
        "mightn't": "might not",  "needn't":   "need not",
        # Subject contractions
        "i'm":      "i am",       "i've":      "i have",
        "i'll":     "i will",     "i'd":       "i would",
        "you're":   "you are",    "you've":    "you have",
        "you'll":   "you will",   "you'd":     "you would",
        "he's":     "he is",      "he'd":      "he would",
        "she's":    "she is",     "she'd":     "she would",
        "it's":     "it is",      "it'd":      "it would",
        "we're":    "we are",     "we've":     "we have",
        "we'll":    "we will",    "we'd":      "we would",
        "they're":  "they are",   "they've":   "they have",
        "they'll":  "they will",  "they'd":    "they would",
        "that's":   "that is",    "that'd":    "that would",
        "there's":  "there is",   "let's":     "let us",
        "who's":    "who is",     "who've":    "who have",
        "what's":   "what is",    "where's":   "where is",
        "when's":   "when is",    "how's":     "how is",
        # Bare suffixes LAST (order matters — must come after full-word forms)
        "'ve":  " have",
        "'re":  " are",
        "'ll":  " will",
        "'d":   " would",
        "'s":   " is",      # catches remaining 's after full forms handled
    }
    for k, v in contractions.items():
        text = text.replace(k, v)
    return text


def step4_remove_punctuation(text: str) -> str:
    """Step 4: Hapus semua tanda baca, hanya sisakan huruf dan spasi."""
    text = re.sub(r"[^a-z\s]", ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def step5_remove_stopwords(text: str) -> str:
    """Step 5: Hapus stopwords (kecuali kata emosi yang dipreserve)."""
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in FINAL_STOPWORDS and len(t) > 1]
    return ' '.join(tokens)


def step6_lemmatize(text: str) -> str:
    """Step 6: Lemmatisasi setiap token dengan POS-aware tagging."""
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t, pos=_get_wordnet_pos(t)) for t in tokens]
    return ' '.join(tokens)


# ── MASTER PIPELINE ───────────────────────────────────────────────────
def preprocess_full_pipeline(text: str) -> dict:
    """
    Jalankan pipeline lengkap dan kembalikan dict berisi:
      - clean_text   : setelah step 1-4 (untuk RoBERTa / teks yang terbaca manusia)
      - final_text   : setelah step 1-6 lengkap (untuk TF-IDF / BM25 / analisis NLP)
    """
    s1 = step1_lowercase(text)
    s2 = step2_remove_noise(s1)
    s3 = step3_expand_contractions(s2)
    s4 = step4_remove_punctuation(s3)   # → clean_text
    s5 = step5_remove_stopwords(s4)
    s6 = step6_lemmatize(s5)            # → final_text
    return {'clean_text': s4, 'final_text': s6}


def apply_pipeline(series: pd.Series) -> pd.DataFrame:
    """Terapkan pipeline ke seluruh Series, kembalikan DataFrame dengan 2 kolom."""
    results = series.apply(preprocess_full_pipeline)
    return pd.DataFrame(results.tolist())


# ── Helpers ───────────────────────────────────────────────────────────
def is_english(text: str) -> bool:
    try:
        return detect(str(text)[:300]) == 'en'
    except LangDetectException:
        return True


def token_count(text: str) -> int:
    if not isinstance(text, str):
        return 0
    return len(text.split())

def clean_for_sbert(text: str) -> str:
    """
    Minimal cleaning for Sentence-BERT input.
    Only: lowercase, remove URLs/HTML/mentions, expand contractions,
    normalize whitespace. Keep punctuation and stopwords intact.
    """
    text = step1_lowercase(text)
    text = step2_remove_noise(text)
    text = step3_expand_contractions(text)
    # normalize leftover special chars but keep sentence punctuation
    text = re.sub(r'[^a-z0-9\s.,!?\'"-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ── Quick sanity check ────────────────────────────────────────────────
sample = "I'm SO tired and stressed... Nothing is going right today!! Check http://stress.com #help"
result = preprocess_full_pipeline(sample)
print('=== Pipeline Sanity Check ===')
print(f'  Input      : {sample}')
print(f'  clean_text : {result["clean_text"]}')
print(f'  final_text : {result["final_text"]}')

---
## 📂 2. Dataset 1 — GoEmotions

**Role dalam pipeline:** Training data untuk fine-tuning model **RoBERTa** (emotion classification)  
**Kolom utama:** `text` → `clean_text`, `final_text`  
**Label:** 27 GoEmotions → 5 project emotion categories

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────
df_ge = pd.read_csv('/kaggle/input/datasets/melisaolivia/nlp-dataset-music-raw/goemotions.csv')
print(f'Shape  : {df_ge.shape}')
print(f'Columns: {df_ge.columns.tolist()}')
df_ge.head(3)

In [ ]:
print('Null values:')
print(df_ge.isnull().sum())
print('\nLabel distribution (top 10):')
print(df_ge['primary_label_name'].value_counts().head(10))

In [ ]:
# ── GoEmotions → 5 Project Emotion Categories ─────────────────────────
# Dipetakan ke: sadness | anger | joy | anxiety | calm
# Sesuai Valence-Arousal mood_mapping di project-context.md

GE_LABEL_MAP = {
    # SADNESS → melancholic mood (valence: -1.0 to -0.5, arousal: -1.0 to 0.0)
    'sadness':        'sadness',
    'grief':          'sadness',
    'remorse':        'sadness',
    'disappointment': 'sadness',

    # ANGER → intense mood (valence: -1.0 to -0.3, arousal: 0.5 to 1.0)
    'anger':          'anger',
    'annoyance':      'anger',
    'disgust':        'anger',
    'disapproval':    'anger',

    # JOY → euphoric mood (valence: 0.5 to 1.0, arousal: 0.3 to 1.0)
    'joy':            'joy',
    'amusement':      'joy',
    'excitement':     'joy',
    'gratitude':      'joy',
    'love':           'joy',
    'admiration':     'joy',
    'pride':          'joy',
    'optimism':       'joy',
    'relief':         'joy',

    # ANXIETY → tense mood (valence: -0.8 to -0.2, arousal: 0.3 to 0.8)
    'fear':           'anxiety',
    'nervousness':    'anxiety',
    'embarrassment':  'anxiety',

    # CALM → peaceful mood (valence: 0.2 to 0.7, arousal: -1.0 to 0.2)
    'approval':       'calm',
    'caring':         'calm',
    'curiosity':      'calm',
    'desire':         'calm',
    'neutral':        'calm',
    'realization':    'calm',
    'surprise':       'calm',
    'confusion':      'calm',
}

print(f'Total label yang dipetakan: {len(GE_LABEL_MAP)} → 5 kategori')
for cat in ['sadness','anger','joy','anxiety','calm']:
    labels = [k for k,v in GE_LABEL_MAP.items() if v == cat]
    print(f'  {cat:10s}: {labels}')

In [ ]:
# ── Cleaning & Preprocessing ──────────────────────────────────────────
df_ge_c = df_ge[['id', 'text', 'primary_label_name']].copy()
df_ge_c.columns = ['sample_id', 'raw_text', 'goemotions_label']

# 1. Drop null
n0 = len(df_ge_c)
df_ge_c.dropna(subset=['raw_text', 'goemotions_label'], inplace=True)
print(f'[1] Drop null         : -{n0 - len(df_ge_c)} baris')

# 2. Drop duplikat pada teks mentah
n0 = len(df_ge_c)
df_ge_c.drop_duplicates(subset=['raw_text'], inplace=True)
print(f'[2] Drop duplikat     : -{n0 - len(df_ge_c)} baris')

# 3. Map ke 5 emotion categories
df_ge_c['emotion_label'] = df_ge_c['goemotions_label'].map(GE_LABEL_MAP)
n0 = len(df_ge_c)
df_ge_c.dropna(subset=['emotion_label'], inplace=True)
print(f'[3] Drop unmapped label: -{n0 - len(df_ge_c)} baris')

# 4. Hapus teks terlalu pendek (< 5 token raw)
n0 = len(df_ge_c)
df_ge_c = df_ge_c[df_ge_c['raw_text'].apply(token_count) >= 5]
print(f'[4] Drop < 5 token    : -{n0 - len(df_ge_c)} baris')

# 5. ★ FULL PREPROCESSING PIPELINE ★
print('[5] Menjalankan preprocessing pipeline...')
pipeline_result = apply_pipeline(df_ge_c['raw_text'])
df_ge_c['clean_text'] = pipeline_result['clean_text']   # step 1-4 (untuk RoBERTa)
df_ge_c['final_text'] = pipeline_result['final_text']   # step 1-6 lengkap
print('    Pipeline selesai.')

# 6. Hapus hasil final_text yang kosong
n0 = len(df_ge_c)
df_ge_c = df_ge_c[df_ge_c['final_text'].str.strip().str.len() > 0]
print(f'[6] Drop final_text kosong: -{n0 - len(df_ge_c)} baris')

# 7. Numeric label ID untuk HuggingFace Trainer
EMOTION_CATEGORIES = sorted(['sadness', 'anger', 'joy', 'anxiety', 'calm'])
LABEL2ID = {l: i for i, l in enumerate(EMOTION_CATEGORIES)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
df_ge_c['label_id'] = df_ge_c['emotion_label'].map(LABEL2ID)

df_ge_c.reset_index(drop=True, inplace=True)

print(f'\n✅ GoEmotions selesai: {len(df_ge_c):,} baris')
print(df_ge_c['emotion_label'].value_counts())

In [ ]:
# ── Contoh hasil preprocessing ────────────────────────────────────────
print('=== Contoh 3 Baris ===')
for _, row in df_ge_c[['raw_text','clean_text','final_text','emotion_label']].head(3).iterrows():
    print(f'  raw_text   : {row["raw_text"]}')
    print(f'  clean_text : {row["clean_text"]}')
    print(f'  final_text : {row["final_text"]}')
    print(f'  emotion    : {row["emotion_label"]}')
    print()

In [ ]:
# ── Simpan label mapping & output CSV ────────────────────────────────
with open('/kaggle/working/label_mapping.json', 'w') as f:
    json.dump({'label2id': LABEL2ID, 'id2label': ID2LABEL}, f, indent=2)
print('Saved: label_mapping.json')

df_ge_out = df_ge_c[[
    'sample_id',       # ID unik
    'raw_text',        # teks asli (referensi)
    'clean_text',      # setelah step 1-4 → gunakan untuk fine-tuning RoBERTa
    'final_text',      # setelah step 1-6 penuh → representasi NLP standar
    'goemotions_label',# label GoEmotions asli
    'emotion_label',   # label 5 kategori project
    'label_id',        # numeric ID untuk Trainer
]]

df_ge_out.to_csv('/kaggle/working/cleaned_goemotions.csv', index=False)
print(f'✅ Saved: cleaned_goemotions.csv — {len(df_ge_out):,} baris')
df_ge_out.head(3)

---
## 📂 4. Dataset 3 — Spotify Lyrics

**Role dalam pipeline:** Basis data mood-to-music untuk **rule-based mapping** (Valence-Arousal)  
**Struktur output:**
- `mood_category` + `emotion_label` → dipakai oleh backend untuk lookup lagu
- `genres_clean` → genre tag dipertahankan untuk query Spotify API
- `final_text` → lirik setelah full pipeline (opsional untuk NLP keyword enrichment)

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────
df_sp = pd.read_csv('/kaggle/input/datasets/melisaolivia/nlp-dataset-music-raw/spotify_lyrics.csv')
# Drop kolom Unnamed
df_sp.drop(columns=[c for c in df_sp.columns if 'Unnamed' in str(c)], inplace=True)
print(f'Shape  : {df_sp.shape}')
print(f'Columns: {df_sp.columns.tolist()}')
df_sp.head(2)

In [ ]:
print('Null values:')
print(df_sp.isnull().sum())
print('\nSample genres:')
print(df_sp['genres'].dropna().head(8).tolist())

In [ ]:
# ── Genre → Mood Category Mapping ─────────────────────────────────────
# Sesuai mood_mapping config di project-context.md

GENRE_MOOD_MAP = {
    # melancholic — sadness (valence: -1.0 to -0.5, arousal: -1.0 to 0.0)
    'indie':              'melancholic',
    'folk':               'melancholic',
    'acoustic':           'melancholic',
    'singer-songwriter':  'melancholic',
    'emo':                'melancholic',
    'blues':              'melancholic',
    'country':            'melancholic',
    'soul':               'melancholic',
    'sad':                'melancholic',
    'melancholic':        'melancholic',

    # intense — anger (valence: -1.0 to -0.3, arousal: 0.5 to 1.0)
    'rock':               'intense',
    'metal':              'intense',
    'punk':               'intense',
    'hard rock':          'intense',
    'hardcore':           'intense',
    'hip hop':            'intense',
    'hip-hop':            'intense',
    'rap':                'intense',
    'trap':               'intense',
    'drill':              'intense',
    'gangsta':            'intense',
    'conscious':          'intense',

    # euphoric — joy (valence: 0.5 to 1.0, arousal: 0.3 to 1.0)
    'pop':                'euphoric',
    'dance':              'euphoric',
    'funk':               'euphoric',
    'edm':                'euphoric',
    'electronic':         'euphoric',
    'disco':              'euphoric',
    'house':              'euphoric',
    'k-pop':              'euphoric',
    'tropical':           'euphoric',
    'upbeat':             'euphoric',

    # tense — anxiety (valence: -0.8 to -0.2, arousal: 0.3 to 0.8)
    'ambient':            'tense',
    'lo-fi':              'tense',
    'lofi':               'tense',
    'dark':               'tense',
    'gothic':             'tense',
    'post-punk':          'tense',
    'alternative':        'tense',
    'experimental':       'tense',
    'industrial':         'tense',

    # peaceful — calm (valence: 0.2 to 0.7, arousal: -1.0 to 0.2)
    'jazz':               'peaceful',
    'classical':          'peaceful',
    'bossa nova':         'peaceful',
    'new age':            'peaceful',
    'meditation':         'peaceful',
    'chill':              'peaceful',
    'r&b':                'peaceful',
    'neo soul':           'peaceful',
    'easy listening':     'peaceful',
    'gospel':             'peaceful',
    'soft rock':          'peaceful',
}

MOOD_TO_EMOTION = {
    'melancholic': 'sadness',
    'intense':     'anger',
    'euphoric':    'joy',
    'tense':       'anxiety',
    'peaceful':    'calm',
}

print(f'Genre keywords dipetakan: {len(GENRE_MOOD_MAP)}')


def assign_mood(genres_str: str) -> str:
    """
    Assign mood category dari semicolon-separated genre string.
    Gunakan majority vote jika ada beberapa genre.
    """
    if not isinstance(genres_str, str) or not genres_str.strip():
        return 'unknown'
    genre_list = [g.strip().lower() for g in genres_str.split(';')]
    scores = {}
    for genre in genre_list:
        for keyword, mood in GENRE_MOOD_MAP.items():
            if keyword in genre:
                scores[mood] = scores.get(mood, 0) + 1
    if not scores:
        return 'unknown'
    return max(scores, key=scores.get)


print('Mood assignment function didefinisikan ✅')

In [ ]:
# ── Cleaning & Preprocessing ──────────────────────────────────────────
df_sp_c = df_sp.copy()

# Standarisasi nama kolom
col_rename = {
    'song':       'title',
    'artists':    'artists_raw',
    'genres':     'genres_raw',
    'lyrics':     'raw_lyrics',
    'song_id':    'song_id',
    'artist_id':  'artist_id',
    'explicit':   'explicit',
}
df_sp_c.rename(columns={k:v for k,v in col_rename.items() if k in df_sp_c.columns}, inplace=True)

# 1. Drop null pada kolom kritis
n0 = len(df_sp_c)
df_sp_c.dropna(subset=['song_id', 'title', 'genres_raw'], inplace=True)
print(f'[1] Drop null (id/title/genre): -{n0 - len(df_sp_c)} baris')

# 2. Bersihkan song_id (hapus prefix 'spotify:track:')
df_sp_c['song_id_clean'] = df_sp_c['song_id'].apply(
    lambda x: re.sub(r'^spotify:track:', '', str(x))
)

# 3. Bersihkan kolom artists (hapus list brackets & quotes)
df_sp_c['artists'] = df_sp_c['artists_raw'].apply(
    lambda x: re.sub(r"[\[\]'\"]", '', str(x)).strip() if pd.notna(x) else ''
)

# 4. Bersihkan dan batasi genres (ambil 3 genre pertama)
df_sp_c['genres_clean'] = df_sp_c['genres_raw'].apply(
    lambda x: '; '.join(g.strip() for g in str(x).split(';')[:3]) if pd.notna(x) else ''
)

# 5. Assign mood_category dari genre
df_sp_c['mood_category'] = df_sp_c['genres_raw'].apply(assign_mood)

# 6. Drop lagu tanpa mood yang dikenali
n0 = len(df_sp_c)
df_sp_c = df_sp_c[df_sp_c['mood_category'] != 'unknown']
print(f'[2] Drop unknown mood         : -{n0 - len(df_sp_c)} baris')

# 7. Map mood → emotion_label (link ke output RoBERTa)
df_sp_c['emotion_label'] = df_sp_c['mood_category'].map(MOOD_TO_EMOTION)

# 8. Bersihkan lirik — hapus artefak scraping
def clean_lyrics_raw(text: str) -> str:
    if not isinstance(text, str):
        return ''
    # Scraping artifacts
    text = re.sub(r'\d*\s*embed\b',         '', text, flags=re.IGNORECASE)  # '1Embed', 'Embed'
    text = re.sub(r'\blyrics?\b',            '', text, flags=re.IGNORECASE)  # 'lyric', 'lyrics'
    text = re.sub(r'You might also like',    '', text, flags=re.IGNORECASE)
    text = re.sub(r'See \w+ lyrics',         '', text, flags=re.IGNORECASE)  # 'See full lyrics'
    text = re.sub(r'\[.*?\]',               '', text)   # [Verse], [Chorus]
    text = re.sub(r'\(.*?\)',               '', text)   # (x2), (repeat)
    text = re.sub(r'\\n',                  ' ', text)   # literal \n
    text = re.sub(r'\s+',                  ' ', text).strip()
    return text

df_sp_c['lyrics_raw_clean'] = df_sp_c['raw_lyrics'].apply(clean_lyrics_raw)

# ── Mismatch detection: flag songs where lyrics may be wrong ──────────
def lyrics_likely_mismatch(title: str, lyrics: str) -> bool:
    """
    Heuristic: if none of the significant title words appear in the
    first 200 chars of the lyrics, flag as potential mismatch.
    Ignore very short titles (1 word) — too many false positives.
    """
    if not isinstance(lyrics, str) or not isinstance(title, str):
        return False
    title_words = [w.lower() for w in title.split() if len(w) > 3]
    if len(title_words) < 2:
        return False
    preview = lyrics[:200].lower()
    return not any(w in preview for w in title_words)

df_sp_c['mismatch_flag'] = df_sp_c.apply(
    lambda r: lyrics_likely_mismatch(r['title'], r['lyrics_raw_clean']), axis=1
)

n_mismatch = df_sp_c['mismatch_flag'].sum()
print(f'[MISMATCH] Flagged {n_mismatch} songs with possible lyrics mismatch')
if n_mismatch > 0:
    print(df_sp_c[df_sp_c['mismatch_flag']][['title', 'artists', 'lyrics_raw_clean']]\
          .head(10).to_string())

# Drop flagged mismatches
n0 = len(df_sp_c)
df_sp_c = df_sp_c[~df_sp_c['mismatch_flag']].copy()
print(f'    Dropped {n0 - len(df_sp_c)} mismatch rows')

# 9. Filter lirik non-Inggris
print('[3] Deteksi bahasa lirik...')
df_sp_c['is_english'] = df_sp_c['lyrics_raw_clean'].apply(
    lambda x: is_english(x[:300]) if token_count(x) > 10 else False
)
n0 = len(df_sp_c)
df_sp_c = df_sp_c[df_sp_c['is_english']]
print(f'    Drop non-English          : -{n0 - len(df_sp_c)} baris')

# 10. Hapus lirik terlalu pendek (< 20 token)
n0 = len(df_sp_c)
df_sp_c = df_sp_c[df_sp_c['lyrics_raw_clean'].apply(token_count) >= 20]
print(f'[4] Drop lyrics < 20 tok      : -{n0 - len(df_sp_c)} baris')

# 11. Dedup pada song_id_clean
n0 = len(df_sp_c)
df_sp_c.drop_duplicates(subset=['song_id_clean'], inplace=True)
print(f'[5] Drop duplikat song_id     : -{n0 - len(df_sp_c)} baris')

# 12. ★ FULL PREPROCESSING PIPELINE pada lirik ★
print('[6] Menjalankan preprocessing pipeline pada lirik...')
pipeline_result = apply_pipeline(df_sp_c['lyrics_raw_clean'])
df_sp_c['clean_text'] = pipeline_result['clean_text']   # step 1-4
df_sp_c['final_text'] = pipeline_result['final_text']   # step 1-6 penuh
print('    Pipeline selesai.')

# 13. Hapus final_text kosong
n0 = len(df_sp_c)
df_sp_c = df_sp_c[df_sp_c['final_text'].str.strip().str.len() > 0]
print(f'[7] Drop final_text kosong    : -{n0 - len(df_sp_c)} baris')

# 14. Build Spotify URL
df_sp_c['spotify_url'] = 'https://open.spotify.com/track/' + df_sp_c['song_id_clean']

df_sp_c.reset_index(drop=True, inplace=True)

print(f'\n✅ Spotify selesai: {len(df_sp_c):,} baris')
print('Mood category distribution:')
print(df_sp_c['mood_category'].value_counts())

In [ ]:
# ── Contoh hasil preprocessing lirik ─────────────────────────────────
print('=== Contoh 2 Lagu ===')
cols = ['title','artists','genres_clean','mood_category','emotion_label','final_text']
for _, row in df_sp_c[cols].head(2).iterrows():
    print(f'  title         : {row["title"]}')
    print(f'  artists       : {row["artists"]}')
    print(f'  genres_clean  : {row["genres_clean"]}')
    print(f'  mood_category : {row["mood_category"]}')
    print(f'  emotion_label : {row["emotion_label"]}')
    print(f'  final_text    : {row["final_text"][:100]}...')
    print()

In [ ]:
# ── Simpan ────────────────────────────────────────────────────────────
df_sp_out = df_sp_c[[
    'song_id_clean',   # Spotify track ID (untuk build URL / API call)
    'title',           # judul lagu
    'artists',         # nama artis (clean)
    'genres_clean',    # ★ tag genre (dipertahankan untuk Spotify API query)
    'mood_category',   # ★ kategori mood (melancholic/intense/euphoric/tense/peaceful)
    'emotion_label',   # ★ label emosi (link ke output RoBERTa → lookup lagu)
    'clean_text',      # lirik setelah step 1-4
    'final_text',      # ★ lirik setelah step 1-6 penuh
    'spotify_url',     # URL playback langsung
]]

df_sp_out.to_csv('/kaggle/working/cleaned_spotify_lyrics.csv', index=False)
print(f'✅ Saved: cleaned_spotify_lyrics.csv — {len(df_sp_out):,} baris')
df_sp_out.head(3)

---
## 📊 5. Summary Report

In [ ]:
print('=' * 70)
print('  DATA CLEANING SUMMARY')
print('=' * 70)

datasets = [
    ('GoEmotions',             df_ge_out,  'cleaned_goemotions.csv'),
    ('EmpatheticDialogues',    df_ed_out,  'cleaned_empathetic_dialogues.csv'),
    ('Spotify Lyrics',         df_sp_out,  'cleaned_spotify_lyrics.csv'),
]

for name, df, fname in datasets:
    print(f'\n📄 {name}')
    print(f'   File    : {fname}')
    print(f'   Rows    : {len(df):,}')
    print(f'   Columns : {df.columns.tolist()}')
    if 'emotion_label' in df.columns:
        print(f'   Labels  : {df["emotion_label"].value_counts().to_dict()}')
    if 'mood_category' in df.columns:
        print(f'   Moods   : {df["mood_category"].value_counts().to_dict()}')

print('\n' + '=' * 70)
print('\n📁 Output files:')
for f in sorted(os.listdir('/kaggle/working')):
    path = f'/kaggle/working/{f}'
    size = os.path.getsize(path) / 1024
    print(f'   {f:50s} {size:>8.1f} KB')

---
## 🗂️ 6. Panduan Penggunaan File Output

### Kolom `final_text` vs `clean_text`

| Kolom | Pipeline | Gunakan untuk |
|-------|----------|---------------|
| `clean_text` | Step 1–4 (lowercase, remove noise, expand contraction, remove punctuation) | Input **RoBERTa** fine-tuning — tokenizer RoBERTa menangani sisanya |
| `final_text` | Step 1–6 + stopwords removal + lemmatization | **TF-IDF / BM25** indexing, analisis NLP, feature extraction |

### 1. `cleaned_goemotions.csv` → RoBERTa Fine-tuning
```python
from datasets import Dataset
import pandas as pd
import json

df = pd.read_csv('cleaned_goemotions.csv')
with open('label_mapping.json') as f:
    mapping = json.load(f)  # {'label2id': {...}, 'id2label': {...}}

# Gunakan clean_text (bukan final_text) — RoBERTa punya tokenizer sendiri
dataset = Dataset.from_pandas(
    df[['clean_text', 'label_id']].rename(columns={'clean_text': 'text', 'label_id': 'labels'})
)
```

### 2. `cleaned_empathetic_dialogues.csv` → BM25 Retrieval Index
```python
from rank_bm25 import BM25Okapi
import pandas as pd

df = pd.read_csv('cleaned_empathetic_dialogues.csv')

# Build index per emotion category (filter untuk relevansi lebih tinggi)
def build_index(emotion: str):
    sub = df[df['emotion_label'] == emotion].reset_index(drop=True)
    corpus = [text.split() for text in sub['final_text']]  # gunakan final_text
    bm25 = BM25Okapi(corpus)
    return bm25, sub

# Retrieve response:
# bm25, sub = build_index('sadness')
# query_tokens = preprocessed_user_input.split()
# best_doc = bm25.get_top_n(query_tokens, sub['response'].tolist(), n=1)[0]
```

### 3. `cleaned_spotify_lyrics.csv` → Rule-Based Mood Lookup
```python
import pandas as pd

df = pd.read_csv('cleaned_spotify_lyrics.csv')

# Lookup lagu berdasarkan emotion_label dari output RoBERTa
def get_playlist(emotion_label: str, n: int = 5) -> list:
    songs = df[df['emotion_label'] == emotion_label]
    sample = songs.sample(min(n, len(songs)))
    return sample[['song_id_clean', 'title', 'artists', 'genres_clean',
                   'mood_category', 'spotify_url']].to_dict('records')

# Contoh: get_playlist('sadness')  → 5 lagu melancholic
```